# TIB final523 scoring on Colab T4 (v1)
Choose a **T4 GPU** runtime, then run the cells in order. Upload the supplied scoring ZIP **and** `start_authorization_scoring.json` when asked. Keep Drive private.

Scores **2088 delivered summaries** of 2092 generation cells (Lead-3, LexRank, BART control, LED ngram3; **no Qwen3**)
with the frozen scorer v2 (ROUGE-1/2/L, BERTScore; source similarity is computed but not reported), then the S/M/L views (ROUGE + BERTScore).
The 4 failed generation cells stay in the denominator with NULL metrics and are never scored. Nothing is generated or selected here.

Scoring has **its own ledger** (`scoring/resource_ledger_scoring.json`) and up to **2 sessions**, the second being the spare:
the generation ledger ended with 3 of its 4 sessions used (one lost), so your decision of 2026-10-25 (`scoring_budget_decision_final523_v1.json`) carved the
scoring budget out of what it left: 10.85 h in total, so generation use plus scoring can never exceed the approved 21 h.
The generation ledger is only read here. The scoring ledger opens with the generation bundle hash, as the generation ledger did.
**Resuming an interrupted scoring run** (policy `scorer_resume_policy_final523_v1.json`, declared by you before any final score existed):
set `RESUME_MODE = 'partial'` in the first cell. Valid checkpoints are reused, only absent ones are scored, 6 reused rows are re-scored as a drift canary
(tolerance 1e-6; a mismatch stops the run), torn `*.partial` files are quarantined and listed, and the checkpoint key binds the compute runtime (CUDA, device,
Python, pinned packages) but not the VM. Corrupt, foreign or mismatched checkpoints stop the run; they are never treated as holes. `'complete'` only replays a
COMPLETE cache. The status cell tells you which case you are in. Scoring work is projected at ~2.4 h; the second session continues a cut-off run.
If the first session is lost or cut off, open a **new** Colab runtime (Disconnect and delete runtime) and run again with `RESUME_MODE = 'partial'`.
The bundle holds sources and references: keep Drive private and never publish the ZIP.

Run once, in order. **There is no cell that closes the session early:** only the last cell does, after the evidence is exported.


In [ ]:
from pathlib import Path
import hashlib, json, os, resource, shutil, subprocess, sys, time, zipfile
import psutil
RESUME_MODE = 'none'   # 'none' fresh | 'partial' continue an interrupted run (pre-run policy) | 'complete' replay a COMPLETE cache
assert RESUME_MODE in ('none', 'partial', 'complete'), RESUME_MODE
RESUME = RESUME_MODE != 'none'
EXPECTED_ZIP_SHA256 = '6ae68b7a0ee95afe0551dac8f0b15352656169ab269c60f8fa58a64d99ebd18b'
EXPECTED_MANIFEST_SHA256 = 'da389b46612f05bd91fd0bea64056afab0e9b27d74d3a9c716a21b00cac58108'
GENERATION_MANIFEST_SHA256 = 'deddb588d008b94e9e531139fc16b38a8b487ec115b92479097373cb2f0a9c69'   # the shared ledger is bound to the GENERATION bundle, not to this one
GENERATION_EVIDENCE_NAME = 'tib-final523-evidence-20261025T143504Z.zip'
GENERATION_EVIDENCE_SHA256 = '0ed836e74df7ae8e60b6795da7d4dff85ee0690baaab2e39d35b1e258bcd5afa'
EXPORT_RESERVE_SECONDS = 0.3 * 3600
SESSION_STARTED = time.monotonic()
SESSION_EPOCH_NS = time.time_ns()
SETUP_CPU_START = resource.getrusage(resource.RUSAGE_CHILDREN)
SETUP_NET_START = psutil.net_io_counters()
SETUP_WALL_START = SESSION_STARTED
WORK = Path('/content/TIB_FINAL523_261024_V1')
BUNDLE = WORK / 'scoring'
SESSION_OPENED = SCORER_READY = GPU_OK = False


## 1. Runtime gate
Checks the GPU **before** Drive or the ledger is touched. A CPU or other GPU runtime stops here and costs nothing.


In [ ]:
# Cheap check before Drive or the shared ledger is touched: a wrong runtime must not cost a session.
try:
    probe = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
                           capture_output=True, text=True, timeout=60)
    gpu_names = [line.strip() for line in probe.stdout.splitlines() if line.strip()] if probe.returncode == 0 else []
except (OSError, subprocess.SubprocessError):
    gpu_names = []
if gpu_names != ['Tesla T4']:
    raise SystemExit('HOLD: the runtime GPU is ' + (', '.join(gpu_names) or 'missing') + ', not one Tesla T4. '
                     'Choose Runtime > Change runtime type > T4 GPU and rerun from the first cell. The ledger was not touched.')
GPU_OK = True
print('GPU gate passed:', gpu_names[0])


## 2. Verify the authorization, the ZIP and the ledgers (read-only)
Without a valid `start_authorization_scoring.json` bound to this bundle nothing is extracted or installed.


In [ ]:
from google.colab import drive, files
BUNDLE.mkdir(parents=True, exist_ok=True)
drive.mount('/content/drive')
STORE = Path('/content/drive/MyDrive/TIB_FINAL523_261024_V1')
GENERATION_DIR = STORE / 'selection'
OUTPUT = STORE / 'scoring'
OUTPUT.mkdir(parents=True, exist_ok=True)
def sha(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for part in iter(lambda: handle.read(4 * 1024 * 1024), b''):
            digest.update(part)
    return digest.hexdigest()
def bootstrap_hold(reason):
    hold = {'status': 'HOLD', 'reason': str(reason), 'scoring_complete': False, 'scoring_start_authorized': False}
    (OUTPUT / ('bootstrap-hold-' + str(time.time_ns()) + '.json')).write_text(json.dumps(hold, indent=2))
uploaded = files.upload()   # choose the supplied scoring ZIP and start_authorization_scoring.json
# Authorization gate first: without a user-made file bound to this exact bundle nothing is extracted or installed.
AUTH = OUTPUT / 'start_authorization_scoring.json'
supplied = [n for n in uploaded if n.endswith('.json')]
if supplied:
    AUTH.write_bytes(uploaded[supplied[0]])
if not AUTH.exists():
    bootstrap_hold('start_authorization_scoring.json missing: not authorized to start')
    raise SystemExit('HOLD: start_authorization_scoring.json missing. Nothing was extracted, installed or opened.')
authorization = json.loads(AUTH.read_text())
required = {'manifest_sha256', 'scoring_readiness_receipt_sha256', 'authorized_by', 'authorized_on', 'statement'}
receipt_hash = str(authorization.get('scoring_readiness_receipt_sha256', '')) if isinstance(authorization, dict) else ''
if not (isinstance(authorization, dict) and required <= set(authorization) and authorization['authorized_by'] == 'user'
        and str(authorization['statement']).strip() and authorization['manifest_sha256'] == EXPECTED_MANIFEST_SHA256
        and len(receipt_hash) == 64 and set(receipt_hash) <= set('0123456789abcdef')):
    bootstrap_hold('start_authorization_scoring.json incomplete, not made by the user, or bound to another bundle')
    raise SystemExit('HOLD: scoring authorization invalid for this bundle.')
archives = [Path(n) for n in uploaded if n.endswith('.zip')]
assert len(archives) == 1, 'Upload exactly the supplied scoring ZIP'
archive = archives[0]
assert sha(archive) == EXPECTED_ZIP_SHA256, 'ZIP differs from the prepared scoring bundle'
with zipfile.ZipFile(archive) as z:
    names = z.namelist()
    assert len(names) == len(set(names)) and z.testzip() is None, 'Invalid ZIP'
    for item in z.infolist():
        assert (BUNDLE / item.filename).resolve().is_relative_to(BUNDLE.resolve()), 'Unsafe path'
        assert (item.external_attr >> 16) & 0o170000 != 0o120000, 'Symlink member'
    z.extractall(BUNDLE)
assert sha(BUNDLE / 'package_manifest.json') == EXPECTED_MANIFEST_SHA256
manifest = json.loads((BUNDLE / 'package_manifest.json').read_text())
assert set(names) == set(manifest['files']) | {'package_manifest.json'}
assert all(sha(BUNDLE / n) == v for n, v in manifest['files'].items()), 'Bundle file hash mismatch'
contract = json.loads((BUNDLE / 'scoring_contract.json').read_text())
assert (contract['generation_bundle_manifest_sha256'] == GENERATION_MANIFEST_SHA256
        and contract['generation_evidence_zip_sha256'] == GENERATION_EVIDENCE_SHA256), 'Contract binds other generation hashes'
# The generation evidence (the scored outputs' origin) must still be intact on Drive.
evidence = STORE / GENERATION_EVIDENCE_NAME
if not evidence.exists() or sha(evidence) != GENERATION_EVIDENCE_SHA256:
    bootstrap_hold('Generation evidence ZIP missing or changed on Drive')
    raise SystemExit('HOLD: ' + GENERATION_EVIDENCE_NAME + ' must be on Drive in the generation folder, unchanged.')
# Scoring ledger: its own budget, carved out of what the generation ledger left. The generation ledger is only read here.
GEN_LEDGER = GENERATION_DIR / 'resource_ledger.json'
LEDGER = OUTPUT / 'resource_ledger_scoring.json'
allocation = json.loads((BUNDLE / 'selection_allocation.json').read_text())
carve = allocation['carved_from_generation']
assert GEN_LEDGER.exists(), 'Generation ledger missing from Drive; it is never recreated'
sys.path.insert(0, str(BUNDLE))
from experiments.scripts.tib_stage_session import digest
from experiments.scripts.tib_final523_scorer_validation import check_scoring_ledger_ready
gen_state = json.loads(GEN_LEDGER.read_text())
assert digest(json.loads((GENERATION_DIR / 'allocation.json').read_text())) == carve['generation_allocation_sha256'], 'Drive generation allocation differs from the carved-out budget'
assert gen_state['bundle_sha256'] == GENERATION_MANIFEST_SHA256, 'Generation ledger is bound to another bundle'
assert gen_state['session_status'] != 'active', 'Generation ledger still shows an active session'
assert gen_state['gpu_seconds'] <= carve['generation_ledger_used']['gpu_seconds'] + 60, 'Generation ledger shows more use than the carved-out budget assumed'
usage = check_scoring_ledger_ready(LEDGER, allocation, GENERATION_MANIFEST_SHA256,
                                   boot_id=Path('/proc/sys/kernel/random/boot_id').read_text().strip())   # raises when bound elsewhere, active on another VM, or no session left
assert RESUME or not any((OUTPUT / 'checkpoints').glob('*.json')), 'Scoring checkpoints exist: set RESUME_MODE to partial (continue) or complete (replay)'
print('Authorization and bundle verified. Sessions used:', usage['sessions_used'], 'of', usage['max_sessions'],
      '| GPU hours used:', round(usage['gpu_hours_used'], 3), 'of', usage['total_gpu_hours'])


## 2b. Status (read-only)
Scored and view checkpoints against the contract, ledger use, and whether an earlier cache can be continued or replayed.


In [ ]:
EXPECTED = contract['expected_scoreable']
def stamp(): return time.strftime('%H:%M:%S')
def checkpoint_count(folder): return len(list((OUTPUT / folder).glob('*.json'))) if (OUTPUT / folder).exists() else 0
def scored(): return checkpoint_count('checkpoints')
def viewed(): return checkpoint_count('view_checkpoints')
ledger_now = json.loads(LEDGER.read_text()) if LEDGER.exists() else {'session_runs': [], 'gpu_seconds': 0.0}   # no scoring session yet
print('[%s] scoring sessions used %d of %d | scoring GPU %.2f of %.2f h (its own ledger)' % (
    stamp(), len(ledger_now['session_runs']), allocation['max_sessions'], ledger_now['gpu_seconds'] / 3600, allocation['total_gpu_hours']))
print('scored checkpoints %d/%d | view checkpoints %d/%d | failed generation cells kept as NULL: %d' % (
    scored(), EXPECTED, viewed(), EXPECTED, contract['expected_failed']))
if scored() or viewed():
    complete = scored() == EXPECTED and viewed() == EXPECTED
    if complete:
        print("Earlier checkpoints found: COMPLETE cache. RESUME_MODE = 'complete' replays it")
    else:
        print("Earlier checkpoints found: PARTIAL (%d scored, %d views of %d). RESUME_MODE = 'partial' continues it: valid checkpoints are "
              "reused, 6 of them are re-scored as a drift canary, and only the absent ones are scored" % (scored(), viewed(), EXPECTED))
    if RESUME_MODE == 'none':
        print("WARNING: RESUME_MODE is 'none'; set it to continue, or move the checkpoint folders aside to start over")
else:
    print('Next: score all', EXPECTED, 'delivered summaries, then the S/M/L views')
partials = [p.name for folder in ('checkpoints', 'view_checkpoints') if (OUTPUT / folder).exists() for p in (OUTPUT / folder).glob('*.partial')]
if partials and RESUME_MODE == 'partial':
    print('%d torn *.partial checkpoint files (a killed write) will be quarantined and their rows scored again' % len(partials))
elif partials:
    print("WARNING: %d torn *.partial checkpoint files (a killed write): only RESUME_MODE = 'partial' can quarantine them" % len(partials))


## 3. Open the ledger session
Uses the generation bundle hash. Refuses when no authorized session is left.


In [ ]:
assert GPU_OK, 'Run the runtime gate cell successfully before opening the ledger session'
sys.path.insert(0, str(BUNDLE))
from experiments.scripts.tib_stage_session import begin_session, record_setup, finish_session
ledger = begin_session(LEDGER, allocation, GENERATION_MANIFEST_SHA256, local_working_root=WORK,
    observation={'boot_id': Path('/proc/sys/kernel/random/boot_id').read_text().strip(),
                 'monotonic': SESSION_STARTED, 'epoch_ns': SESSION_EPOCH_NS, 'logical_cpu_count': os.cpu_count()})
SESSION_LIMIT_SECONDS = ledger['session_limit_seconds']
SESSION_STARTED = ledger['session_started_monotonic']
SESSION_DEADLINE_SECONDS = SESSION_LIMIT_SECONDS * (1 - allocation['session_headroom_fraction'])
SESSION_OPENED = True
print('Session opened. Granted hours:', SESSION_LIMIT_SECONDS / 3600, '| scoring deadline hours:', SESSION_DEADLINE_SECONDS / 3600,
      '| sessions used:', len(ledger['session_runs']), 'of', allocation['max_sessions'])


## 4. Pinned runtime
Proven pip-less venv bootstrap, same pins as the Pilot40 scorer run.


In [ ]:
ENV = WORK/'venv'
PYTHON = ENV/'bin/python'
RUN_ENV = dict(os.environ, USE_TF='0', USE_FLAX='0', TOKENIZERS_PARALLELISM='false',
               PYTHONPATH=str(BUNDLE), HF_HUB_DISABLE_TELEMETRY='1')
assert shutil.disk_usage('/content').free >= 12*1024**3, 'Need at least 12 GiB working disk before setup.'
def execute(command, log_name):
    remaining = SESSION_LIMIT_SECONDS-(time.monotonic()-SESSION_STARTED)
    assert remaining>0, 'Session guard reached'
    log_path = OUTPUT/log_name
    try:
        with log_path.open('w') as log:
            result = subprocess.run(command, cwd=BUNDLE, env=RUN_ENV, stdout=log,
                                    stderr=subprocess.STDOUT, timeout=remaining)
        print('exit', result.returncode, '|', log_path)
        print(log_path.read_text(errors='replace')[-5000:])
        return result.returncode
    except Exception as error:
        bootstrap_hold(error)
        raise
# An unsuccessful ensurepip step can leave bin/python present. Always repair the
# venv structure without deleting it or bootstrapping pip inside the target.
created = execute([sys.executable,'-m','venv','--system-site-packages','--without-pip',str(ENV)],
                  'create-venv.log')
if created:
    bootstrap_hold('venv --without-pip failed; see create-venv.log')
assert created==0, 'HOLD: venv creation failed. Export evidence using the last cell.'
probe = "import json,sys; from pathlib import Path; assert sys.prefix != sys.base_prefix; assert Path(sys.prefix).absolute() == Path(sys.argv[1]).absolute(); print(json.dumps({'executable':sys.executable,'prefix':sys.prefix,'base_prefix':sys.base_prefix,'python':sys.version}))"
verified = execute([str(PYTHON),'-c',probe,str(ENV)],'venv-identity.log')
if verified:
    bootstrap_hold('Target interpreter is not the intended venv; see venv-identity.log')
assert verified==0, 'HOLD: target venv identity failed.'
# pip >=22.3 officially supports installing into a pip-less target interpreter.
# Use the base installer only as an installer, never as the scorer interpreter.
setup = execute([sys.executable,'-m','pip','--python',str(PYTHON),'install',
                 '-r',str(BUNDLE/'requirements-colab-t4.txt')], 'install.log')
if setup:
    bootstrap_hold('Pinned dependency installation failed; see install.log (base pip needs --python support)')
assert setup==0, 'HOLD: dependency setup failed. Export evidence using the last cell.'


## 5. Scorer assets and torch preflight
Only the two pinned scorer models are acquired. Setup time, CPU and transfer are charged to the ledger once, here.


In [ ]:
assert SESSION_OPENED
ASSETS = WORK / 'scorer-assets'
asset_code = """
from pathlib import Path
from experiments.scripts.tib_benchmark_scorer_assets import acquire
from experiments.scripts.tib_benchmark_scorer import validate_assets
from experiments.scripts.rq2_tib_benchmark_metrics import BERT_MODEL,BERT_REVISION,SBERT_MODEL,SBERT_REVISION
root=Path(__import__('sys').argv[1])
if not all((root/k/'asset_manifest.json').exists() for k in ('bert','sentence')):
    acquire(root)
for k,m,r in [('bert',BERT_MODEL,BERT_REVISION),('sentence',SBERT_MODEL,SBERT_REVISION)]:
    validate_assets(root/k,model=m,revision=r)
print('Actual pinned scorer asset hashes verified')
"""
asset_exit = execute([str(PYTHON), '-c', asset_code, str(ASSETS)], 'assets.log')
if asset_exit:
    bootstrap_hold('Scorer assets incomplete/invalid; see assets.log')
assert asset_exit == 0, 'HOLD: assets. Export evidence, then close the session with the last cell.'
torch_probe = ("import json, torch; assert torch.cuda.is_available(), 'CUDA unavailable in the venv'; "
               "name = torch.cuda.get_device_name(0); assert name == 'Tesla T4', name; "
               "print(json.dumps({'torch': torch.__version__, 'device': name}))")
torch_exit = execute([str(PYTHON), '-c', torch_probe], 'torch-preflight.log')
if torch_exit:
    bootstrap_hold('torch/CUDA preflight failed in the venv; see torch-preflight.log')
assert torch_exit == 0, 'HOLD: torch preflight. Export evidence, then close the session with the last cell.'
usage = resource.getrusage(resource.RUSAGE_CHILDREN)
net = psutil.net_io_counters()
setup_cpu = (usage.ru_utime + usage.ru_stime) - (SETUP_CPU_START.ru_utime + SETUP_CPU_START.ru_stime)
setup_transfer = (net.bytes_recv + net.bytes_sent) - (SETUP_NET_START.bytes_recv + SETUP_NET_START.bytes_sent)
record_setup(LEDGER, cpu_seconds=setup_cpu, transfer_bytes=max(0, setup_transfer), wall_seconds=time.monotonic() - SETUP_WALL_START)
SCORER_READY = True


## 6. Score
Fresh scoring is the default. Order: long sources first, each source's four profiles together, then the views. Progress prints every 30 s with checkpoint count, rate, ETA and the session budget left. The runner refuses to start when the ledger session is not active.


In [ ]:
assert SESSION_OPENED and SCORER_READY, 'Run the session, setup and asset cells successfully first'
assert 'scored' in globals(), 'Run the status cell first'
POLL_SECONDS = 30
FINISHED = ('completed', 'completed_with_warning', 'failed')

def stamp(): return time.strftime('%H:%M:%S')
def left_hours(): return (SESSION_LIMIT_SECONDS - (time.monotonic() - SESSION_STARTED)) / 3600

def read_log(log_path, offset):
    with log_path.open('r', errors='replace') as handle:
        handle.seek(offset)
        text = handle.read()
        return text, handle.tell()

def stop_process(process):
    """SIGTERM, then SIGKILL after 30 s, so an old process can never keep writing next to a resumed run."""
    process.terminate()
    try:
        process.wait(timeout=30)
    except subprocess.TimeoutExpired:
        process.kill()
        process.wait()

def run_logged(command, log_name, label, counter, total, limit_seconds, unit='rec'):
    """Run a subprocess, printing progress every POLL_SECONDS; checkpoints survive any interruption.

    counter() returns how many of `total` units are checkpointed; limit_seconds stops the subprocess at the session guard.
    """
    assert limit_seconds > 0, 'Session guard reached'
    log_path = OUTPUT / log_name
    started, offset, next_report, start_count, pending = time.monotonic(), 0, 0, counter(), ''
    with log_path.open('w') as log:
        process = subprocess.Popen(command, cwd=BUNDLE, env=RUN_ENV, stdout=log, stderr=subprocess.STDOUT)
        try:
            while process.poll() is None:
                time.sleep(2)
                elapsed = time.monotonic() - started
                if elapsed > limit_seconds:
                    stop_process(process)
                    bootstrap_hold('Session guard reached during ' + label)
                    print('[%s] %s: session guard reached; checkpoints kept' % (stamp(), label))
                    break
                if elapsed >= next_report:
                    next_report = elapsed + POLL_SECONDS
                    fresh, offset = read_log(log_path, offset)
                    lines = (pending + fresh).split('\n')
                    pending = lines.pop()   # an incomplete last line waits for the next poll
                    shown = 0
                    for line in lines:
                        try:
                            item = json.loads(line)
                        except ValueError:
                            if shown < 5:
                                print('   log:', line[:300])
                                shown += 1
                            continue
                        if isinstance(item, dict) and item.get('status') in FINISHED:
                            print('   [%s] %s %s' % (stamp(), item.get('record_id'), item['status']))
                    done = counter()
                    rate = (done - start_count) / (elapsed / 3600) if done > start_count and elapsed else 0.0
                    eta = '%.2f h' % ((total - done) / rate) if rate else 'n/a'
                    print('[%s] %-8s %4d/%d %s (%.1f%%) | %.1f %s/h | ETA %s | elapsed %.2f h | session left %.2f h | free disk %.1f GiB' % (
                        stamp(), label, done, total, unit, 100 * done / total, rate, unit, eta,
                        elapsed / 3600, left_hours(), shutil.disk_usage('/content').free / 1024**3))
            process.wait()
        except BaseException:
            stop_process(process)
            print('[%s] %s interrupted: checkpoints kept; rerun with RESUME=True. The record in flight may stay started and is never retried.' % (stamp(), label))
            raise
    print('[%s] %s exit %s | %.2f h | log %s' % (stamp(), label, process.returncode, (time.monotonic() - started) / 3600, log_path.name))
    print(log_path.read_text(errors='replace')[-1500:])
    return process.returncode

def left_hours(): return (SESSION_DEADLINE_SECONDS - EXPORT_RESERVE_SECONDS - (time.monotonic() - SESSION_STARTED)) / 3600
remaining_hours = (SESSION_DEADLINE_SECONDS - (time.monotonic() - SESSION_STARTED) - EXPORT_RESERVE_SECONDS) / 3600
assert remaining_hours > 0.25, 'Too little session time left; do not start scoring.'
command = [str(PYTHON), '-m', 'experiments.scripts.tib_pilot40_scorer_runner', '--stage', 'final523',
           '--bundle', str(BUNDLE), '--output', str(OUTPUT), '--assets', str(ASSETS), '--ledger', str(LEDGER),
           '--wall-hours', str(remaining_hours)]   # the runner stops early enough to leave time to export
if RESUME_MODE == 'complete':
    command.append('--resume')
elif RESUME_MODE == 'partial':
    command.append('--resume-partial')
launch_ns = time.time_ns()
bootstrap_hold('Scorer subprocess started; awaiting current actual receipt')
exit_code = run_logged(command, 'scorer-' + str(launch_ns) + '.log', 'scoring', lambda: scored() + viewed(),
                       2 * EXPECTED, remaining_hours * 3600 + 300, unit='ckpt')   # the runner stops itself first; the kill is a 5 min backstop
print('[%s] scored %d/%d | views %d/%d' % (stamp(), scored(), EXPECTED, viewed(), EXPECTED))
if exit_code not in (0, 2):
    bootstrap_hold('Scorer subprocess failed before a complete receipt; inspect logs')
receipt_path = OUTPUT / 'scorer_final523_receipt.json'
if receipt_path.exists() and exit_code in (0, 2):
    current = json.loads(receipt_path.read_text())
    if int(current.get('run_id', 0)) >= launch_ns:
        print(json.dumps({k: current.get(k) for k in ('delivered_status', 'views_status', 'reasons',
              'successful_metric_records', 'expected_scoreable', 'partial_resume_used', 'reused_rows', 'fresh_rows', 'reused_failed_rows', 'mixed_runtime',
              'run_ids', 'quarantined', 'canary')}, indent=2))
    else:
        bootstrap_hold('Receipt does not belong to this execution; a prior PASS cannot certify it')
else:
    bootstrap_hold('No actual scorer receipt produced')
print('Do not infer PASS from install/download completion or local tests.')


### Progress from a terminal
Colab Terminal (bash):
```bash
D=/content/drive/MyDrive/TIB_FINAL523_261024_V1/scoring; while true; do clear; date; echo "scored: $(ls $D/checkpoints 2>/dev/null | wc -l)/2088"; echo "views: $(ls $D/view_checkpoints 2>/dev/null | wc -l)/2088"; f=$(ls -t $D/scorer-*.log 2>/dev/null | head -1); [ -n "$f" ] && tail -n 6 "$f"; nvidia-smi --query-gpu=utilization.gpu,memory.used,memory.total --format=csv,noheader; sleep 15; done
```
`scored` counts per-output checkpoints of the delivered summaries; `views` counts the S/M/L checkpoints.


## 7a. Export the evidence ZIP (also after a HOLD)
Read-only for the ledger and safe to repeat. Sources and references are left out. Keep the ZIP private.


In [ ]:
from datetime import datetime, timezone
# Safe to run any number of times, before or after the session is closed: it only reads the ledger.
stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
export = STORE / ('tib-final523-scoring-evidence-' + stamp + '.zip')
with zipfile.ZipFile(export, 'w', zipfile.ZIP_DEFLATED) as z:
    for path in sorted(OUTPUT.rglob('*')):
        if path.is_file() and '__pycache__' not in path.parts:
            z.write(path, 'evidence/' + path.relative_to(OUTPUT).as_posix())
    for path in sorted(BUNDLE.rglob('*')):
        # Sources and references stay out of the returned evidence; their hash is in the manifest.
        if path.is_file() and '__pycache__' not in path.parts and path.name != 'scoring_outputs.jsonl':
            z.write(path, 'bundle/' + path.relative_to(BUNDLE).as_posix())
    if LEDGER.exists():
        z.write(LEDGER, 'ledger/resource_ledger_at_export.json')
    if 'ASSETS' in globals():
        for key in ('bert', 'sentence'):
            manifest_path = ASSETS / key / 'asset_manifest.json'
            if manifest_path.exists():
                z.write(manifest_path, 'scorer-assets/' + key + '/asset_manifest.json')
print('ZIP:', export, '| bytes:', export.stat().st_size, '| SHA256:', sha(export))
files.download(str(export))


## 7b. Close the session
Run **once, after 7a**. It closes the ledger session only while it is active; rerunning does nothing. Send back the ZIP and `resource_ledger_after_scoring.json`.


In [ ]:
# The only cell that closes the session. It acts once, only while the ledger session is active, and is safe to rerun.
state = json.loads(LEDGER.read_text())
if state['session_status'] == 'active':
    try:
        sys.path.insert(0, str(BUNDLE))   # a fresh kernel or VM has neither the path nor the import from the session cell
        from experiments.scripts.tib_stage_session import finish_session
        finish_session(LEDGER, status='scoring_evidence_exported')
        print('Session closed in the shared ledger.')
    except (ValueError, ImportError) as error:
        print('Session NOT closed (' + str(error) + '); the ledger keeps the conservative accounting. Do not rerun scoring.')
else:
    print('Session already closed (' + str(state['session_status']) + '); nothing to do.')
final_copy = STORE / 'resource_ledger_after_scoring.json'
final_copy.write_bytes(LEDGER.read_bytes())
files.download(str(final_copy))   # small text-free file: send it back with the evidence ZIP for the audit
